# Enhanced LLM Librarian Demo

This notebook demonstrates the enhanced LLMLibrarian with ChromaDB vector storage and tool calling capabilities.

In [ ]:
# Import the enhanced librarian
import sys
sys.path.append('../src')

from LLMLibrarian.librarian_v2 import EnhancedLibrarian

## Initialize the Librarian

Configure the librarian with your OpenAI-compatible API endpoint. This example uses a local LLM server.

In [ ]:
# Initialize with local LLM server
librarian = EnhancedLibrarian(
    api_base_url="http://localhost:11434/v1",  # Ollama or similar
    api_key="sk-dummy",  # Not used by local servers
    embedding_model_name="all-MiniLM-L6-v2",
    chunk_size=1000,
    chunk_overlap=200
)

## Index a Directory

Let's index a sample project directory. The librarian will:
- Scan all files recursively
- Extract text from supported file types
- Create embeddings and store in ChromaDB
- Cache processed files to avoid reprocessing

In [ ]:
# Index a directory (change this to your project path)
project_path = "../src/LLMLibrarian"  # Example: the LLMLibrarian source itself

stats = librarian.process_directory(project_path)
print(f"\nIndexing complete!")
print(f"Total files: {stats['total_files']}")
print(f"Cached files: {stats['cached_files']}")
print(f"Processed files: {stats['processed_files']}")

## Query Without Tools

Basic queries use vector similarity search to find relevant code chunks.

In [ ]:
# Ask a question about the codebase
response = librarian.query(
    "What is the main purpose of this project?",
    use_tools=False
)
print(response)

In [ ]:
# Ask about specific functionality
response = librarian.query(
    "How does the text extraction work?",
    use_tools=False
)
print(response)

## Query With Tools

Enable tool calling to let the AI use system commands for more precise searches.

In [ ]:
# Find specific files
response = librarian.query(
    "Find all Python files that contain 'chromadb' in their name or content",
    use_tools=True
)
print(response)

In [ ]:
# Search for patterns
response = librarian.query(
    "Search for all class definitions in the codebase",
    use_tools=True
)
print(response)

In [ ]:
# Get file information
response = librarian.query(
    "What's the size and structure of the main librarian module?",
    use_tools=True
)
print(response)

## Session History

The librarian tracks all queries and responses in a session.

In [ ]:
# View recent session history
history = librarian.get_session_history(limit=5)
for entry in history:
    print(f"[{entry.get('type', 'unknown')}] {entry.get('content', '')[:100]}...")

## Advanced Queries

Combine vector search with tool usage for complex queries.

In [ ]:
# Complex architectural question
response = librarian.query(
    "Explain the architecture of this project. What are the main components and how do they interact?",
    use_tools=True
)
print(response)

In [ ]:
# Find potential issues
response = librarian.query(
    "Are there any TODO comments or potential issues marked in the code?",
    use_tools=True
)
print(response)

## Reindex After Changes

The smart caching system only reprocesses changed files.

In [ ]:
# Reindex to pick up any changes
stats = librarian.process_directory(project_path)
print(f"\nReindexing complete!")
print(f"Total files: {stats['total_files']}")
print(f"Cached files: {stats['cached_files']} (unchanged)")
print(f"Processed files: {stats['processed_files']} (new/modified)")